In [ ]:
# ============================================================
# CELL 0. Imports and file loading
# ============================================================

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

# ----------------------------
# File path
# ----------------------------
DATA_DIR = Path(r"C:\Users\jdelhoyo\PhD\Study cases\Genissiat\RV Characterization\repo-github\data")
FILE_NAME = "RV_For_RF4_Index_2.csv"
FILE_PATH = DATA_DIR / FILE_NAME

# Output directory
OUT_DIR = DATA_DIR / "ELE_analysis_outputs"
OUT_DIR.mkdir(exist_ok=True)

# ----------------------------
# Load data
# sep=None tries to infer delimiter; useful if the .csv is actually tab-separated
# ----------------------------
df = pd.read_csv(FILE_PATH, sep=None, engine="python")

# Clean column names: strip leading/trailing spaces
df.columns = df.columns.str.strip()

print("Loaded file:", FILE_PATH)
print("Shape:", df.shape)
print("\nAvailable columns:")
for c in df.columns:
    print("-", c)

# ----------------------------
# Define core columns
# ----------------------------
FLW_COL = "Dead_Wood"             # floodplain large wood
CLW_COL = "LW_Presence"           # in-channel large wood
ELE_COL = "ELE (EffectLatExp)"    # effective lateral exposure

# Optional diagnostic columns
VALLEY_COL = "ValleyConfinIndex"
LAT_COL = "Lat_Connectivity"

# Other potentially relevant predictors
OPTIONAL_PREDICTORS = [
    "ValleyConfinIndex",
    "Lat_Connectivity",
    "SPI / Width",
    "Gradient (%)",
    "Sinuosity",
    "Width_Mean",
    "Basal_Area (m2/ha)",
    "StructuralIndex",
    "P50_Height",
    "Height_IQR",
    "Distance to outlet (km)"
]

# Check required columns
required = [FLW_COL, CLW_COL, ELE_COL]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# Convert core columns to numeric
for c in [FLW_COL, CLW_COL, ELE_COL, VALLEY_COL, LAT_COL] + OPTIONAL_PREDICTORS:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# Drop rows without core variables
df_core = df.dropna(subset=[FLW_COL, CLW_COL, ELE_COL]).copy()

# Ensure ordinal classes are integer
df_core[FLW_COL] = df_core[FLW_COL].astype(int)
df_core[CLW_COL] = df_core[CLW_COL].astype(int)

print("\nCore data shape after dropping missing FLW/CLW/ELE:", df_core.shape)
print(df_core[[FLW_COL, CLW_COL, ELE_COL]].head())

In [ ]:
# ============================================================
# CELL 1. Descriptive statistics and diagnostics:
# Is ELE measuring something different?
# ============================================================

def descriptive_stats(series):
    x = series.dropna()
    return pd.Series({
        "n": x.size,
        "min": x.min(),
        "Q1": x.quantile(0.25),
        "median": x.median(),
        "mean": x.mean(),
        "Q3": x.quantile(0.75),
        "max": x.max(),
        "std": x.std()
    })

desc_vars = [ELE_COL]
if VALLEY_COL in df_core.columns:
    desc_vars.append(VALLEY_COL)
if LAT_COL in df_core.columns:
    desc_vars.append(LAT_COL)

desc_table = pd.DataFrame({v: descriptive_stats(df_core[v]) for v in desc_vars}).T
print("\nDescriptive statistics:")
display(desc_table)

desc_table.to_csv(OUT_DIR / "ELE_descriptive_statistics.csv", index=True)

# ----------------------------
# Distribution of Lat_Connectivity
# ----------------------------
if LAT_COL in df_core.columns:
    lat_counts = df_core[LAT_COL].value_counts(dropna=False).sort_index()
    lat_pct = df_core[LAT_COL].value_counts(normalize=True, dropna=False).sort_index() * 100

    lat_table = pd.DataFrame({
        "count": lat_counts,
        "percent": lat_pct.round(2)
    })

    print("\nLat_Connectivity distribution:")
    display(lat_table)

    n_less_4 = (df_core[LAT_COL] < 4).sum()
    pct_less_4 = 100 * n_less_4 / df_core[LAT_COL].notna().sum()

    print(f"\nCases with Lat_Connectivity < 4: {n_less_4} ({pct_less_4:.2f}%)")

    lat_table.to_csv(OUT_DIR / "LatConnectivity_distribution.csv", index=True)

# ----------------------------
# Spearman correlations: ELE and ValleyConfinIndex versus predictors
# ----------------------------
def spearman_pair(data, x_col, y_col):
    tmp = data[[x_col, y_col]].dropna()
    if tmp.shape[0] < 3:
        return np.nan, np.nan, tmp.shape[0]
    if tmp[x_col].nunique() < 2 or tmp[y_col].nunique() < 2:
        return np.nan, np.nan, tmp.shape[0]
    rho, p = stats.spearmanr(tmp[x_col], tmp[y_col])
    return rho, p, tmp.shape[0]

predictors = [p for p in OPTIONAL_PREDICTORS if p in df_core.columns and p != ELE_COL]

rows = []
for pred in predictors:
    rho_ele, p_ele, n_ele = spearman_pair(df_core, ELE_COL, pred)

    if VALLEY_COL in df_core.columns:
        rho_val, p_val, n_val = spearman_pair(df_core, VALLEY_COL, pred)
    else:
        rho_val, p_val, n_val = np.nan, np.nan, np.nan

    rows.append({
        "Predictor": pred,
        "rho_ELE": rho_ele,
        "p_ELE": p_ele,
        "n_ELE": n_ele,
        "rho_ValleyConfinIndex": rho_val,
        "p_ValleyConfinIndex": p_val,
        "n_ValleyConfinIndex": n_val,
        "difference_abs_rho": abs(rho_ele) - abs(rho_val) if pd.notna(rho_ele) and pd.notna(rho_val) else np.nan
    })

corr_table = pd.DataFrame(rows)
print("\nSpearman correlations: ELE and ValleyConfinIndex versus selected variables")
display(corr_table)

corr_table.to_csv(OUT_DIR / "ELE_vs_other_variables_spearman.csv", index=False)

# ----------------------------
# Specific ELE vs ValleyConfinIndex diagnostic
# ----------------------------
if VALLEY_COL in df_core.columns:
    rho_ev, p_ev, n_ev = spearman_pair(df_core, ELE_COL, VALLEY_COL)
    print(f"\nELE vs ValleyConfinIndex: Spearman rho = {rho_ev:.3f}, p = {p_ev:.4g}, n = {n_ev}")

    plt.figure(figsize=(7, 6))
    if LAT_COL in df_core.columns:
        sc = plt.scatter(
            df_core[VALLEY_COL],
            df_core[ELE_COL],
            c=df_core[LAT_COL],
            alpha=0.8
        )
        cbar = plt.colorbar(sc)
        cbar.set_label("Lat_Connectivity")
    else:
        plt.scatter(df_core[VALLEY_COL], df_core[ELE_COL], alpha=0.8)

    min_val = np.nanmin([df_core[VALLEY_COL].min(), df_core[ELE_COL].min()])
    max_val = np.nanmax([df_core[VALLEY_COL].max(), df_core[ELE_COL].max()])
    plt.plot([min_val, max_val], [min_val, max_val], linestyle="--", linewidth=1)

    plt.xlabel("ValleyConfinIndex")
    plt.ylabel("ELE (EffectLatExp)")
    plt.title(f"ELE vs ValleyConfinIndex\nSpearman rho = {rho_ev:.3f}, p = {p_ev:.4g}")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "ELE_vs_ValleyConfinIndex.png", dpi=300)
    plt.show()

# ----------------------------
# ELE by Lat_Connectivity
# ----------------------------
if LAT_COL in df_core.columns:
    plt.figure(figsize=(7, 5))
    sns.boxplot(data=df_core, x=LAT_COL, y=ELE_COL)
    sns.stripplot(data=df_core, x=LAT_COL, y=ELE_COL, color="black", alpha=0.6, jitter=True)
    plt.xlabel("Lat_Connectivity")
    plt.ylabel("ELE (EffectLatExp)")
    plt.title("ELE distribution by Lat_Connectivity")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "ELE_by_LatConnectivity.png", dpi=300)
    plt.show()